# 5 · TWDB Groundwater — Well Inventory, Water Levels & Quality

Reads the watershed boundaries from notebook 1, discovers **Texas Water Development Board
(TWDB)** **Groundwater Database (GWDB)** wells inside them via the TWDB ArcGIS FeatureServer,
then fetches their water-level and water-quality **measurements** from TWDB's nightly full-state
bulk file — the FeatureServer only carries well *inventory* (location, aquifer, flags), not the
actual time series. Primary source:
<https://www.twdb.texas.gov/groundwater/data/index.asp>.

## Step 1 — Imports and setup

In [1]:
import geopandas as gpd
import hvplot.pandas  # noqa: F401  (registers .hvplot on DataFrames — used by the trend chart)
import pandas as pd

import geoviews as gv
import geoviews.tile_sources as gvts

from _helpers import (
    init_session,
    save_dataframe,
    load_dataframe,
    show,
    categorical_colors,
    make_legend_clickable,
    fetch_gwdb_wells,
    fetch_gwdb_zip,
    fetch_gwdb_members,
    tidy_gwdb_water_levels,
    tidy_gwdb_water_quality,
    coverage,
    trend_by_group,
)
from _helpers.twdb import WATER_LEVEL_MEMBERS, WATER_LEVEL_USECOLS, WATER_QUALITY_MEMBERS, WATER_QUALITY_USECOLS

gv.extension("bokeh")
S = init_session()

USGS API key loaded.


## Step 2 — Discover wells

Queries the GWDB well-inventory ArcGIS FeatureServer within the watersheds' bounding box, then
keeps only wells **within** the watershed polygons — same bbox-then-spatial-join pattern as
notebooks 3-4.

> **Caching note:** the ArcGIS query bypasses the on-disk request cache (`cache/`) HyRiver calls
> use — so we treat the saved `twdb_wells.parquet` itself as a **backup cache**
> (`load_dataframe`, one-week freshness window): if it's still fresh, we load it and skip the
> network call entirely.

In [2]:
boundaries_path = S.data_dir / "hydrography" / "huc8_watersheds.parquet"
if not boundaries_path.exists():
    raise FileNotFoundError(
        f"{boundaries_path} not found — run notebook 1 (1_usgs_hydrography) first."
    )
watersheds_gdf = gpd.read_parquet(boundaries_path)
bbox = list(watersheds_gdf.total_bounds)  # [min_lon, min_lat, max_lon, max_lat]; reused below

wells_path = S.data_dir / "twdb_groundwater" / "twdb_wells.parquet"
wells_in_area = load_dataframe(wells_path, max_age_days=7)
if wells_in_area is None:
    wells_gdf = fetch_gwdb_wells(bbox)
    wells_in_area = gpd.sjoin(
        wells_gdf,
        watersheds_gdf[["huc8", "name", "geometry"]],
        predicate="within",
        how="inner",
    )
    print(f"{len(wells_gdf)} GWDB wells in the bounding box; {len(wells_in_area)} within the watersheds.")
    save_dataframe(wells_in_area, wells_path)
show(wells_in_area[["StateWellNumber", "CountyName", "WaterLevelObservationType", "WaterQualityAvailable", "name"]])

using cached data/twdb_groundwater/twdb_wells.parquet (1906 rows, < 7 days old)


,StateWellNumber,CountyName,WaterLevelObservationType,WaterQualityAvailable,name
72,8631901,Starr,Historical,Y,Los Olmos
73,8631902,Starr,Miscellaneous Measurements,N,Los Olmos
74,8631903,Starr,None,N,Los Olmos
75,8632101,Starr,Miscellaneous Measurements,Y,Los Olmos
78,8632201,Starr,Miscellaneous Measurements,N,Los Olmos
79,8632202,Starr,Miscellaneous Measurements,Y,Los Olmos
80,8632203,Starr,Miscellaneous Measurements,N,Los Olmos
81,8632204,Starr,None,Y,Los Olmos
82,8632301,Starr,Historical,Y,Los Olmos
83,8632302,Starr,Miscellaneous Measurements,N,Los Olmos


## Step 3 — Download the nightly GWDB bulk file

The ArcGIS layer above is inventory-only; the actual water-level/quality measurements live in a
nightly full-state zip, cached locally for a week (like this project's other request caches) so
re-running the notebook doesn't re-download ~81 MB every time. If both `twdb_water_levels` and
`twdb_water_quality` below are already fresh, we skip the download entirely — there's no need to
even check the zip if neither of its contents is going to be read this run.

In [3]:
levels_path = S.data_dir / "twdb_groundwater" / "twdb_water_levels.parquet"
quality_path = S.data_dir / "twdb_groundwater" / "twdb_water_quality.parquet"
twdb_water_levels = load_dataframe(levels_path, max_age_days=7)
twdb_water_quality = load_dataframe(quality_path, max_age_days=7)

well_ids = set(wells_in_area["StateWellNumber"].dropna())
huc8_by_well = dict(zip(wells_in_area["StateWellNumber"], wells_in_area["huc8"]))

if twdb_water_levels is None or twdb_water_quality is None:
    zip_path = fetch_gwdb_zip(S.repo_root / "data_temp" / "gwdb_download.zip")
    print(f"{zip_path} ({zip_path.stat().st_size:,} bytes); {len(well_ids)} wells to filter for")
else:
    print("water levels and water quality are both cached — skipping the bulk-file download")

using cached data/twdb_groundwater/twdb_water_levels.parquet (10133 rows, < 7 days old)
using cached data/twdb_groundwater/twdb_water_quality.parquet (2092 rows, < 7 days old)
water levels and water quality are both cached — skipping the bulk-file download


## Step 4 — Water levels

Filters the four Major/Minor/Combination/OtherUnassigned water-level files down to our wells,
streamed in chunks (the files are large statewide extracts). Skipped when `twdb_water_levels`
already came from the cache above.

In [4]:
if twdb_water_levels is None:
    water_levels_raw = fetch_gwdb_members(zip_path, WATER_LEVEL_MEMBERS, WATER_LEVEL_USECOLS, well_ids)
    show(water_levels_raw.head())  # peek at the raw GWDB file shape before we tidy it

In [5]:
if twdb_water_levels is None:
    twdb_water_levels = tidy_gwdb_water_levels(water_levels_raw, huc8_by_well)
    show(twdb_water_levels.head())
    save_dataframe(twdb_water_levels, levels_path)

## Step 5 — Water quality

Same filtering approach for the water-quality files; classification reuses `classify_parameter`
on GWDB's `ParameterCode` (confirmed in the sandbox exploration to reuse USGS-style codes).
Skipped when `twdb_water_quality` already came from the cache above.

In [6]:
if twdb_water_quality is None:
    water_quality_raw = fetch_gwdb_members(zip_path, WATER_QUALITY_MEMBERS, WATER_QUALITY_USECOLS, well_ids)
    show(water_quality_raw.head())  # peek at the raw GWDB file shape before we tidy it

In [7]:
if twdb_water_quality is None:
    twdb_water_quality = tidy_gwdb_water_quality(water_quality_raw, huc8_by_well)
    show(twdb_water_quality.head())
    save_dataframe(twdb_water_quality, quality_path)

print(f"Wells by data type (of {len(wells_in_area)}):")
print(pd.Series({
    "water_level": twdb_water_levels["monitoring_location_id"].nunique(),
    "water_quality": twdb_water_quality["monitoring_location_id"].nunique(),
}).to_string())

Wells by data type (of 1906):
water_level      1333
water_quality     675


## Step 6 — Map wells by data availability

Wells with water-level records, water-quality records, or both, over the watershed outlines.

In [8]:
level_ids = set(twdb_water_levels["monitoring_location_id"])
quality_ids = set(twdb_water_quality["monitoring_location_id"])
DATA_TYPE_COLORS = categorical_colors(["water_level", "water_quality"])
watershed_outlines = gv.Path(watersheds_gdf).opts(color="black", line_width=1.5)

wells_map = gvts.EsriWorldTopo * watershed_outlines
for label, ids in [("water_level", level_ids), ("water_quality", quality_ids)]:
    subset = wells_in_area[wells_in_area["StateWellNumber"].isin(ids)]
    if len(subset) == 0:
        continue
    wells_map = wells_map * gv.Points(
        subset, vdims=["StateWellNumber", "CountyName"], label=label,
    ).opts(color=DATA_TYPE_COLORS[label], size=7, line_color="white", tools=["hover"])

wells_map = wells_map.opts(
    data_aspect=1,
    title="TWDB GWDB wells by data type (click legend to toggle)",
    legend_position="right",
    hooks=[make_legend_clickable],
)
wells_map

:Overlay
   .WMTS.I               :WMTS   [Longitude,Latitude]
   .Path.I               :Path   [Longitude,Latitude]   (objectid,tnmid,metasourceid,sourcedatadesc,sourceoriginator,sourcefeatureid,loaddate,referencegnis_ids,areaacres,areasqkm,states,huc8,name,globalid,shape_Length,shape_Area)
   .Points.Water_level   :Points   [Longitude,Latitude]   (StateWellNumber,CountyName)
   .Points.Water_quality :Points   [Longitude,Latitude]   (StateWellNumber,CountyName)

## Step 7 — Data availability

In [9]:
show(coverage(twdb_water_levels, "datetime"))

,monitoring_location_id,priority_group,n,start,end
0,8631901,water_level,9,1950-11-01,1962-06-25
1,8631902,water_level,1,1950-11-01,1950-11-01
2,8632101,water_level,1,1950-11-02,1950-11-02
3,8632201,water_level,1,1950-11-02,1950-11-02
4,8632202,water_level,1,1950-11-05,1950-11-05
5,8632203,water_level,1,1950-11-02,1950-11-02
6,8632301,water_level,6,1956-01-10,1963-08-13
7,8632302,water_level,1,1950-10-31,1950-10-31
8,8632401,water_level,1,1950-12-15,1950-12-15
9,8632402,water_level,1,1951-01-10,1951-01-10


In [10]:
show(coverage(twdb_water_quality, "datetime"))

,monitoring_location_id,priority_group,n,start,end
985,8859411,conductivity,9,1989-06-05,1989-06-19
175,8731918,dissolved_oxygen,1,2009-09-15,2009-09-15
245,8737702,dissolved_oxygen,1,2009-08-26,2009-08-26
255,8738101,dissolved_oxygen,1,2009-08-25,2009-08-25
283,8740203,dissolved_oxygen,1,2009-09-17,2009-09-17
377,8746701,dissolved_oxygen,1,2009-08-04,2009-08-04
411,8747802,dissolved_oxygen,1,2009-08-04,2009-08-04
488,8754835,dissolved_oxygen,1,2009-08-05,2009-08-05
507,8754949,dissolved_oxygen,1,2009-08-05,2009-08-05
528,8755504,dissolved_oxygen,1,2009-08-06,2009-08-06


## Step 8 — Trends (Mann–Kendall + Sen's slope)

Per well × priority-parameter, annual **median** — water levels and quality samples are both
irregular series, same treatment as notebooks 3-4's water-quality trends.

In [11]:
level_trends = trend_by_group(
    twdb_water_levels, ["monitoring_location_id", "priority_group"], "datetime", "water_elevation_ft", agg="median"
)
quality_trends = trend_by_group(
    twdb_water_quality, ["monitoring_location_id", "priority_group"], "datetime", "value", agg="median"
)
twdb_trends = pd.concat(
    [level_trends.assign(data_type="water_level"), quality_trends.assign(data_type="water_quality")],
    ignore_index=True,
)
twdb_trends["significant"] = twdb_trends["p"] < 0.05
save_dataframe(twdb_trends, S.data_dir / "twdb_groundwater" / "twdb_trends.parquet")
show(twdb_trends.round({"p": 4, "slope": 4}))

saved 2539 rows → data/twdb_groundwater/twdb_trends.parquet (+ .csv)


,monitoring_location_id,priority_group,trend,p,slope,intercept,n,data_type,significant
0,8631901,water_level,increasing,0.0187,0.3025,182.091250,8,water_level,True
1,8631902,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
2,8632101,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
3,8632201,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
4,8632202,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
5,8632203,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
6,8632301,water_level,no trend,0.2207,3.9775,350.765000,5,water_level,False
7,8632302,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
8,8632401,water_level,insufficient,NaN,NaN,NaN,1,water_level,False
9,8632402,water_level,insufficient,NaN,NaN,NaN,1,water_level,False


In [12]:
trend_chart = twdb_trends.dropna(subset=["slope"]).hvplot.bar(
    x="priority_group", y="slope", by="data_type",
    hover_cols=["monitoring_location_id", "trend", "p", "significant"],
    frame_height=360, rot=40,
    ylabel="Sen's slope (per year)", xlabel="",
    title="TWDB well trend rates by priority parameter (Sen's slope)", legend="top_right",
).opts(active_tools=[])
trend_chart

:Bars   [priority_group,data_type]   (slope,monitoring_location_id,trend,p,significant)

## What's next

Well inventory, water levels, water quality, and trends are saved under `data/twdb_groundwater/`.
A future shared display notebook can compare USGS/TCEQ/TWDB trends side by side, and TWDB's
coastal surface-water data (waterdatafortexas.org, relevant to the South Laguna Madre watershed)
remains a candidate for a later round.